# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

In [ ]:
# TASK TYPE: Ranking & Scoring (Lane 4 Opportunity Score)
# EXPLANATION: We are calculating a continuous opportunity/residual score for each visible page,
# then ranking them from highest to lowest opportunity. This outputs an ordered priority queue 
# for editorial review to address metadata, snippets, or intent mismatches.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

In [17]:
# TARGET PROXY: is_underperforming (Binary: 1 or 0)
# HOW IT IS MEASURED: A page is flagged as underperforming (1) if it has high visibility (impressions_90d >= 100)
# but its actual click-through rate (ctr) falls below the specific median CTR for its own content_type.
# This prevents penalizing structural layouts (like comparison articles) that naturally net lower baseline CTRs.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

In [18]:
# METRICS: Precision@20 and Precision@50
# EXPLANATION: Of the top 20 or 50 pages our model places in the priority queue, what fraction are actually 
# underperforming? Checking both ensures our top-priority batch is highly reliable for editors with limited hours.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [19]:
# UNIT OF ANALYSIS: One row = One unique pseudonymized content item (page) belonging to a client.
import pandas as pd
import numpy as np

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

# Print dataset dimensions to verify
print(f"Dataset shape: {df.shape}")

# FIX: Map the local content-type thresholds to ensure the label matches our non-linear thesis.
content_medians = df[df["impressions_90d"] >= 100].groupby("content_type")["ctr"].median()
df["median_ctr_threshold"] = df["content_type"].map(content_medians)

df["is_underperforming"] = np.where(
    (df["impressions_90d"] >= 100) & (df["ctr"] < df["median_ctr_threshold"]), 1, 0
)

# Display a slice showing the unit of analysis and target
features = [
    "content_type", "main_intent", "competition_level", "position_tier",
    "impressions_90d", "sessions_90d", "content_age_days", "days_since_last_update"
]
df[features + ["median_ctr_threshold", "ctr", "is_underperforming"]].head(5)

Dataset shape: (30000, 44)


,content_type,main_intent,competition_level,position_tier,impressions_90d,sessions_90d,content_age_days,days_since_last_update,median_ctr_threshold,ctr,is_underperforming
0,keyword article,transactional,HIGH,striking,3803,17,187,20,0.15,0.76,0
1,keyword article,informational,LOW,page_3_5,15320,9,445,25,0.15,0.05,1
2,keyword article,informational,LOW,page_3_5,12581,11,141,20,0.15,0.09,1
3,keyword article,commercial,LOW,page_1,11751,78,463,22,0.15,0.49,0
4,keyword article,informational,LOW,page_3_5,19140,145,263,14,0.15,0.13,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In [ ]:
# WHY ML WINS HERE: A simple hand-written rule cannot scale across dozens of cohorts. 
# Lane 4 dictates that a page's performance must be evaluated relative to its position_tier 
# and format. ML acts as a multi-dimensional residual/gap analysis engine, untangling non-linear 
# interactions between content_type, competition_level, and search depth to reveal true underperformance.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.